# 00 - Colab Setup And Research Configuration

            Run this first. It installs dependencies, mounts Google Drive, reads your Colab secrets, logs in to Hugging Face and W&B, and writes a shared run configuration for the later notebooks.

            Required Colab secrets:

            - `HF_WRITE_ACCESS`
            - `WANDB_KEY`

            Recommended runtime: Colab Pro GPU. A100 is ideal, L4 works for the default Qwen 0.5B/1.5B runs.

In [3]:
from pathlib import Path
import os
import subprocess

repo = Path("/content/AutoRegressive-Bhasha")

if not repo.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/ritwikraha/AutoRegressive-Bhasha.git",
            str(repo),
        ],
        check=True,
    )

os.chdir(repo / "empty-negations")
print("Working directory:", Path.cwd())

Working directory: /content/AutoRegressive-Bhasha/empty-negations


In [6]:
from pathlib import Path
import os, sys, json, subprocess, textwrap

def find_repo_root():
    try:
        import google.colab  # type: ignore  # noqa: F401
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except Exception:
        pass

    candidates = [
        Path.cwd(),
        Path("/content/empty-negations"),
        Path("/content/drive/MyDrive/ocn_empty_negations"),
        Path("/content/drive/MyDrive/AutoRegressive-Bhasha/empty-negations"),
    ]
    for candidate in candidates:
        if (candidate / "src/ocn").exists():
            return candidate
    repo_url = os.environ.get("OCN_REPO_URL", "")
    if repo_url:
        target = Path("/content/empty-negations")
        if not target.exists():
            subprocess.run(["git", "clone", repo_url, str(target)], check=True)
        return target
    raise FileNotFoundError(
        "Could not find the empty-negations repo. Run this notebook from the repo, "
        "copy it to /content/drive/MyDrive/ocn_empty_negations, or set OCN_REPO_URL."
    )

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
print("Repo:", REPO_ROOT)

Repo: /content/AutoRegressive-Bhasha/empty-negations


In [7]:
import sys, subprocess

requirements = REPO_ROOT / "requirements-colab.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
print("Installed:", requirements)

Installed: /content/AutoRegressive-Bhasha/empty-negations/requirements-colab.txt


In [8]:
from huggingface_hub import HfApi
from ocn.colab_utils import login_huggingface, login_wandb, make_colab_paths, utc_timestamp

HF_TOKEN = login_huggingface("HF_WRITE_ACCESS")
api = HfApi(token=HF_TOKEN)
HF_OWNER = api.whoami()["name"]

RUN_ID = utc_timestamp()
PROJECT_NAME = "ocn_empty_negations"
HF_DATASET_PREFIX = "ocn-empty-negations"

paths = make_colab_paths(PROJECT_NAME)
run = login_wandb(
    project="ocn-empty-negations",
    name=f"setup-{RUN_ID}",
    config={"hf_owner": HF_OWNER, "run_id": RUN_ID},
)

CONFIG = {
    "run_id": RUN_ID,
    "hf_owner": HF_OWNER,
    "hf_private": False,
    "hf_prompt_repo": f"{HF_OWNER}/{HF_DATASET_PREFIX}-prompts",
    "hf_generation_repo": f"{HF_OWNER}/{HF_DATASET_PREFIX}-generations",
    "hf_detection_repo": f"{HF_OWNER}/{HF_DATASET_PREFIX}-detection",
    "hf_reward_pairs_repo": f"{HF_OWNER}/{HF_DATASET_PREFIX}-reward-pairs",
    "hf_reward_scores_repo": f"{HF_OWNER}/{HF_DATASET_PREFIX}-reward-scores",
    "drive_project_root": str(paths.project_root),
    "drive_data_root": str(paths.data_root),
    "drive_figure_root": str(paths.figure_root),
    "default_prompt_limit": 96,
    "default_seeds": [1, 2],
}

config_path = paths.project_root / "ocn_colab_config.json"
config_path.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
print(json.dumps(CONFIG, indent=2))

run.finish()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ritwik to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


{
  "run_id": "20260816T061520Z",
  "hf_owner": "ritwikraha",
  "hf_private": false,
  "hf_prompt_repo": "ritwikraha/ocn-empty-negations-prompts",
  "hf_generation_repo": "ritwikraha/ocn-empty-negations-generations",
  "hf_detection_repo": "ritwikraha/ocn-empty-negations-detection",
  "hf_reward_pairs_repo": "ritwikraha/ocn-empty-negations-reward-pairs",
  "hf_reward_scores_repo": "ritwikraha/ocn-empty-negations-reward-scores",
  "drive_project_root": "/content/drive/MyDrive/ocn_empty_negations",
  "drive_data_root": "/content/drive/MyDrive/ocn_empty_negations/artifacts/data",
  "drive_figure_root": "/content/drive/MyDrive/ocn_empty_negations/artifacts/figures",
  "default_prompt_limit": 96,
  "default_seeds": [
    1,
    2
  ]
}
